
# Thí nghiệm 04 — Xây dựng kiến trúc mô hình Electricity

## 1. Khởi tạo môi trường và cấu hình mô hình

### 1.1. Mục tiêu

Notebook này định nghĩa các kiến trúc mạng nơ-ron phục vụ thí nghiệm dự báo chuỗi thời gian Electricity.

Bốn mô hình được xây dựng gồm:

1. LSTM — mô hình hồi quy tuần tự làm baseline.
2. PatchTST-based Koopformer — kết hợp Transformer và toán tử Koopman.
3. Autoformer-based Koopformer — kết hợp phân rã chuỗi thời gian, Transformer và Koopman.
4. Informer-based Koopformer — kết hợp Transformer Encoder và Koopman.

Ngoài các kiến trúc, notebook định nghĩa các hàm tạo cửa sổ trượt, tính loss, huấn luyện và đánh giá để tái sử dụng trong notebook Training.

### 1.2. Dữ liệu đầu vào

Dữ liệu được tải từ:

`data/electricity_train_test.pt`

Tệp đã được tạo trong `01_electricity_data.ipynb`.

| Thành phần | Kích thước |
|---|---|
| Train | (2800, 6) |
| Test | (700, 6) |
| Số kênh đầu vào | 6 |
| Kiểu dữ liệu | PyTorch Tensor |

Dữ liệu đã được chia theo thời gian và chuẩn hóa Min-Max ở bước chuẩn bị dữ liệu.

### 1.3. Cấu hình chung

| Tham số | Giá trị |
|---|---|
| Random seed | 7 |
| Model dimension | 96 |
| LSTM hidden dimension | 48 |
| Epochs | 4000 |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Thiết bị | CUDA nếu khả dụng, ngược lại CPU |

Các giá trị này là cấu hình được khai báo trong mã nguồn hiện tại.

### 1.4. Phạm vi

Notebook chủ yếu định nghĩa mô hình và các hàm hỗ trợ.

Quá trình huấn luyện chính thức, lưu checkpoint và tổng hợp metric được thực hiện trong `03_electricity_training.ipynb`.

Notebook không thực hiện huấn luyện mô hình tại đây.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from pathlib import Path

# 1. Reproducibility
SEED = 7

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Locate Electricity Dataset
candidates = [
    Path.cwd() / "data" / "electricity_train_test.pt",
    Path.cwd() / "experiments" / "04_electricity" / "data" / "electricity_train_test.pt",
    Path.cwd().parent / "experiments" / "04_electricity" / "data" / "electricity_train_test.pt"
]

DATA_PATH = next(
    (p.resolve() for p in candidates if p.is_file()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Cannot find electricity_train_test.pt"
    )

# 3. Load Preprocessed Dataset
bundle = torch.load(
    DATA_PATH,
    map_location="cpu",
    weights_only=False
)

train_data = bundle["train"]
test_data = bundle["test"]

channels = bundle["channels"]
INPUT_DIM = train_data.shape[1]

# 4. Model Hyperparameters
D_MODEL = 96
LSTM_HIDDEN_DIM = 48

EPOCHS = 4000
LEARNING_RATE = 3e-4
LYAP_WEIGHT = 0.1

# 5. Validate Dataset
assert train_data.shape == (2800, 6)
assert test_data.shape == (700, 6)
assert INPUT_DIM == 6

assert torch.isfinite(train_data).all()
assert torch.isfinite(test_data).all()

# 6. Print Information
print("=== DEEPKOOPFORMER — ELECTRICITY ===")

print("\nDevice:", DEVICE)
print("Dataset:", DATA_PATH)

print("\nTrain shape:", train_data.shape)
print("Test shape:", test_data.shape)

print("\nInput dimension:", INPUT_DIM)
print("Channels:", channels)

print("\nD_MODEL:", D_MODEL)
print("LSTM hidden dimension:", LSTM_HIDDEN_DIM)

print("\nEpochs:", EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("Lyapunov weight:", LYAP_WEIGHT)

print("\nData SHA256:", bundle["data_hash"])


## 2. Toán tử Koopman ổn định nghiêm ngặt

### 2.1. Mục đích

Toán tử Koopman được sử dụng để mô hình hóa phép biến đổi tuyến tính trong không gian biểu diễn tiềm ẩn.

Với biểu diễn đầu vào $\mathbf{z}$, phép biến đổi được mô tả bởi:

$$
\mathbf{z}_{next}=K\mathbf{z}
$$

Trong mã nguồn, các mẫu được biểu diễn dưới dạng vector hàng, nên phép tính tương ứng là:

`z_next = z @ K.T`

### 2.2. Cấu trúc toán tử

Ma trận Koopman được xây dựng theo dạng:

$$
K=U\operatorname{diag}(\sigma)V^\top
$$

Trong đó:

- $U$ và $V$ được trực giao hóa bằng phép phân rã QR.
- $\sigma$ là vector các giá trị kỳ dị được ràng buộc.
- $K$ là ma trận biến đổi trong không gian tiềm ẩn.

Các giá trị kỳ dị được xác định bởi:

$$
\sigma_i=\rho_{\max}\operatorname{sigmoid}(s_i)
$$

Với $\rho_{\max}=0.99$, ta có:

$$
0<\sigma_i<0.99
$$

Do đó chuẩn phổ của ma trận thỏa mãn:

$$
\|K\|_2<0.99<1
$$

Điều này bảo đảm toán tử không khuếch đại chuẩn Euclid của vector đầu vào.

### 2.3. Các thành phần triển khai

Hàm `orthogonalize()` sử dụng `torch.linalg.qr()` để tạo ma trận trực giao.

Class `StrictStableKoopmanOperator` gồm:

- `U_raw`: ma trận tham số dùng để xây dựng $U$.
- `V_raw`: ma trận tham số dùng để xây dựng $V$.
- `S_raw`: tham số tạo các giá trị kỳ dị.
- `rho_max`: giới hạn trên của các giá trị kỳ dị.

Hàm `forward()` trả về biểu diễn sau biến đổi, ma trận Koopman và vector giá trị kỳ dị đã ràng buộc.

### 2.4. Vai trò trong thực nghiệm

Toán tử được sử dụng trong ba biến thể Koopformer dựa trên PatchTST, Autoformer và Informer.

Số chiều toán tử phụ thuộc vào kích thước biểu diễn đầu ra của backbone tương ứng.

In [ ]:
# 1. Orthogonalization
def orthogonalize(W: torch.Tensor) -> torch.Tensor:
    Q, R = torch.linalg.qr(W)
    return Q


# 2. Strict Stable Koopman Operator
class StrictStableKoopmanOperator(nn.Module):

    def __init__(
        self,
        latent_dim: int,
        rho_max: float = 0.99
    ):
        super().__init__()

        # Learnable matrices
        self.U_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.V_raw = nn.Parameter(
            torch.randn(latent_dim, latent_dim)
        )

        self.S_raw = nn.Parameter(
            torch.randn(latent_dim)
        )

        # Upper bound for singular values
        self.rho_max = rho_max

    def forward(self, z: torch.Tensor):

        # Step 1: Orthogonal matrices
        U = orthogonalize(self.U_raw)
        V = orthogonalize(self.V_raw)

        # Step 2: Constrained singular values
        Sigma = (
            torch.sigmoid(self.S_raw)
            * self.rho_max
        )

        # Step 3: Construct Koopman operator
        K = U @ torch.diag(Sigma) @ V.T

        # Step 4: Apply Koopman transformation
        z_next = z @ K.T

        return z_next, K, Sigma


## 3. Mã hóa vị trí bằng hàm Sin và Cos

### 3.1. Mục đích

Transformer không tự biểu diễn thứ tự thời gian của các token. Vì vậy, mô hình sử dụng Positional Encoding để bổ sung thông tin vị trí vào biểu diễn đầu vào.

Class `SinCosPosEnc` xây dựng các vector mã hóa vị trí cố định bằng hàm sin và cos.

### 3.2. Công thức

Với vị trí $pos$ và chiều biểu diễn $i$:

$$
PE_{(pos,2i)}
=
\sin\left(
\frac{pos}{10000^{2i/d_{\mathrm{model}}}}
\right)
$$

$$
PE_{(pos,2i+1)}
=
\cos\left(
\frac{pos}{10000^{2i/d_{\mathrm{model}}}}
\right)
$$

Trong đó $d_{\mathrm{model}}=96$ trong cấu hình mô hình hiện tại.

### 3.3. Phương pháp triển khai

Class thực hiện:

1. Tạo vector chỉ số vị trí.
2. Tính các hệ số tần số.
3. Xây dựng ma trận mã hóa bằng Sin và Cos.
4. Lưu ma trận bằng `register_buffer`.
5. Cộng mã hóa vị trí vào input embeddings trong `forward()`.

Các giá trị mã hóa vị trí không phải là tham số được cập nhật trực tiếp bởi optimizer.

### 3.4. Vai trò

Thành phần này được tái sử dụng trong các backbone dựa trên PatchTST, Autoformer và Informer.

In [ ]:
class SinCosPosEnc(nn.Module):

    def __init__(
        self,
        d_model: int,
        max_len: int = 10000
    ):
        super().__init__()

        # 1. Tạo vị trí: 0, 1, 2, ..., max_len - 1
        pos = torch.arange(
            max_len
        ).float().unsqueeze(1)

        # 2. Tính các hệ số tần số
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-np.log(10000.0) / d_model)
        )

        # 3. Khởi tạo ma trận Positional Encoding
        pe = torch.zeros(
            max_len,
            d_model
        )

        # 4. Mã hóa vị trí bằng sin và cos
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)

        # 5. Lưu tensor vào model
        self.register_buffer("pe", pe)

    def forward(self, x):

        # Cộng Positional Encoding vào Input Embeddings
        return x + self.pe[:x.size(1)]


## 4. Patch Embedding cho chuỗi thời gian

### 4.1. Mục đích

Patch Embedding chuyển chuỗi thời gian đa biến thành các vector đặc trưng có số chiều cố định để đưa vào Transformer Encoder.

Class `PatchEmbed1D` sử dụng tích chập một chiều (`Conv1d`) để thực hiện phép chiếu đặc trưng.

### 4.2. Dữ liệu đầu vào và đầu ra

Dữ liệu đầu vào có dạng:

`(batch_size, sequence_length, input_dim)`

Trước phép tích chập, tensor được chuyển thành:

`(batch_size, input_dim, sequence_length)`

Sau `Conv1d` và phép chuyển vị, đầu ra có dạng:

`(batch_size, number_of_patches, d_model)`

Với kernel size $P$, stride $S$ và không sử dụng padding, số patch được xác định bởi:

$$
N_p=
\left\lfloor
\frac{L-P}{S}
\right\rfloor+1
$$

Trong đó $L$ là độ dài chuỗi đầu vào.

### 4.3. Cấu hình thực nghiệm

Ở các backbone sử dụng Patch Embedding, stride được đặt bằng patch length.

Khi chiều dài chuỗi đầu vào bằng patch length, phép tích chập tạo ra một patch duy nhất.

Điều này cần được phân biệt với cơ chế chia một chuỗi dài thành nhiều patch chồng lấp hoặc không chồng lấp.

### 4.4. Vai trò

`PatchEmbed1D` được sử dụng trong:

- `PatchTST_Backbone`.
- `InformerSparse` khi `patch_len > 1`.

In [ ]:
class PatchEmbed1D(nn.Module):

    def __init__(
        self,
        in_ch: int,
        d_model: int,
        patch_len: int,
        stride: int
    ):
        super().__init__()

        # 1. Convolution để tạo patch embeddings
        self.conv = nn.Conv1d(
            in_channels=in_ch,
            out_channels=d_model,
            kernel_size=patch_len,
            stride=stride
        )

    def forward(self, x):

        # Input: (Batch, Time, Channels)
        # Conv1d yêu cầu: (Batch, Channels, Time)
        x = x.permute(0, 2, 1)

        # 2. Patch Embedding
        x = self.conv(x)

        # Output: (Batch, N_patches, d_model)
        x = x.permute(0, 2, 1)

        return x


## 5. Bộ trích xuất đặc trưng dựa trên PatchTST

### 5.1. Mục đích

`PatchTST_Backbone` chuyển chuỗi Electricity đa biến thành một vector biểu diễn tiềm ẩn phục vụ phép biến đổi Koopman.

Backbone kết hợp Patch Embedding, Positional Encoding, Transformer Encoder và CLS Token.

### 5.2. Các thành phần

1. **Patch Embedding:** ánh xạ chuỗi thời gian thành các vector có số chiều `d_model`.
2. **CLS Token:** tham số có thể học, được thêm vào đầu chuỗi token.
3. **Positional Encoding:** bổ sung thông tin vị trí cho các token.
4. **Transformer Encoder:** sử dụng cơ chế self-attention để biến đổi biểu diễn.
5. **CLS Representation:** lấy đầu ra của CLS Token làm biểu diễn tổng hợp.

### 5.3. Cấu hình kiến trúc

| Tham số | Giá trị |
|---|---|
| Input dimension | 6 |
| Model dimension | 96 |
| Transformer Encoder layers | 3 |
| Attention heads | 4 |
| Feedforward dimension | 96 |
| Patch stride | Bằng patch length |
| Output dimension | 96 |

### 5.4. Luồng dữ liệu

Với cấu hình `seq_len = patch_len`, dữ liệu được xử lý như sau:

`(B, P, 6)` → `(B, 1, 96)` → `(B, 2, 96)` → `(B, 96)`

Trong đó:

- $B$: batch size.
- $P$: patch length.
- Token thứ hai là patch embedding.
- Token thứ nhất là CLS Token.

### 5.5. Giới hạn triển khai

Cách triển khai sử dụng một patch duy nhất khi `seq_len = patch_len`, vì vậy cơ chế attention xử lý CLS Token và một patch token.

Đây là đặc điểm cụ thể của backbone hiện tại, không nên mặc nhiên xem là triển khai đầy đủ của kiến trúc PatchTST gốc.


In [ ]:
class PatchTST_Backbone(nn.Module):

    def __init__(
        self,
        input_dim: int,
        seq_len: int,
        patch_len: int,
        d_model: int = 96,
        num_layers: int = 3,
        num_heads: int = 4,
        dim_ff: int = 96
    ):
        super().__init__()

        # 1. Patch Embedding
        self.patch = PatchEmbed1D(
            in_ch=input_dim,
            d_model=d_model,
            patch_len=patch_len,
            stride=patch_len
        )

        # 2. Positional Encoding
        n_patches = int(np.ceil(seq_len / patch_len))

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_patches + 1
        )

        # 3. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 4. Learnable CLS Token
        self.cls = nn.Parameter(
            torch.randn(1, 1, d_model)
        )

    def forward(self, x):

        # (B, seq_len, input_dim)
        # -> (B, n_patches, d_model)
        x = self.patch(x)

        # Thêm CLS Token vào đầu chuỗi
        cls = self.cls.expand(
            x.size(0), -1, -1
        )

        x = torch.cat(
            [cls, x],
            dim=1
        )

        # Positional Encoding + Transformer Encoder
        x = self.encoder(
            self.pos(x)
        )

        # Lấy representation của CLS Token
        return x[:, 0]


## 6. Mô hình kết hợp PatchTST và toán tử Koopman

### 6.1. Kiến trúc tổng thể

Class `Koopformer_PatchTST` kết hợp ba thành phần:

1. `PatchTST_Backbone`: trích xuất biểu diễn tiềm ẩn.
2. `StrictStableKoopmanOperator`: thực hiện phép biến đổi tuyến tính có ràng buộc ổn định.
3. `Linear Forecasting Head`: giải mã biểu diễn thành các giá trị dự báo.

Luồng xử lý:

**Input → PatchTST Backbone → Latent Representation → Koopman Operator → Linear Decoder → Prediction**

### 6.2. Biểu diễn tiềm ẩn

Backbone ánh xạ chuỗi đầu vào sang vector tiềm ẩn:

$$
\mathbf{z}\in\mathbb{R}^{96}
$$

Toán tử Koopman biến đổi vector thành:

$$
\mathbf{z}_{next}=K\mathbf{z}
$$

Với ma trận $K\in\mathbb{R}^{96\times96}$.

### 6.3. Dự báo

Bộ giải mã tuyến tính ánh xạ từ 96 chiều sang:

$$
H\times6
$$

Trong đó $H$ là Forecast Horizon.

Đầu ra được trải phẳng thành vector dự báo cho sáu kênh Electricity.

### 6.4. Giá trị trả về

Ở chế độ dự báo thông thường, mô hình trả về `pred`.

Khi `return_latents=True`, mô hình trả về:

- `pred`: giá trị dự báo.
- `z`: biểu diễn trước Koopman.
- `z_next`: biểu diễn sau Koopman.

Hai biểu diễn tiềm ẩn được sử dụng để tính thành phần chính quy hóa Lyapunov trong quá trình huấn luyện.


In [ ]:
class Koopformer_PatchTST(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=96
    ):
        super().__init__()

        # 1. PatchTST Backbone
        self.backbone = PatchTST_Backbone(
            input_dim=input_dim,
            seq_len=seq_len,
            patch_len=patch_len,
            d_model=d_model
        )

        # 2. Strict Stable Koopman Operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # 3. Forecasting Head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # 1. Trích xuất latent representation
        z = self.backbone(x)

        # 2. Biến đổi latent bằng Koopman Operator
        z_next, K, Sigma = self.koop(z)

        # 3. Dự báo các bước tương lai
        pred = self.fc(z_next)

        # 4. Trả về latent nếu cần tính Lyapunov Loss
        if return_latents:
            return pred, z, z_next

        return pred


## 7. Phân rã chuỗi thời gian

### 7.1. Mục đích

Class `SeriesDecomp` phân tách chuỗi đầu vào thành thành phần xu hướng và phần dư nhằm phục vụ backbone dựa trên Autoformer.

### 7.2. Phương pháp

Xu hướng được ước lượng bằng phép trung bình trượt:

$$
X_{\mathrm{trend}}=
\operatorname{MovingAverage}(X)
$$

Phần dư được xác định bởi:

$$
X_{\mathrm{residual}}
=
X-X_{\mathrm{trend}}
$$

Do đó:

$$
X=X_{\mathrm{trend}}+X_{\mathrm{residual}}
$$

Trong code, phần dư được đặt tên là `seasonal`. Tuy nhiên, phép trừ xu hướng không tự chứng minh rằng phần dư chỉ chứa thành phần mùa vụ.

### 7.3. Cách triển khai

Class sử dụng `nn.AvgPool1d` với:

- Kernel size: `k`.
- Stride: 1.
- Padding: `k // 2`.

Kích thước tensor được chuyển từ `(B, T, C)` sang `(B, C, T)` để thực hiện trung bình trượt, sau đó chuyển về định dạng ban đầu.

Khi kernel size chẵn, code cắt bớt timestep nếu chiều dài đầu ra lớn hơn chiều dài đầu vào.

### 7.4. Vai trò

Thành phần này được sử dụng bởi `SimpleAutoformer`.

Trong mô hình hiện tại, kernel size được truyền từ tham số `patch_len`.


In [ ]:
class SeriesDecomp(nn.Module):

    def __init__(self, k: int = 3):
        super().__init__()

        # Moving Average để trích xuất Trend
        self.avg = nn.AvgPool1d(
            kernel_size=k,
            stride=1,
            padding=k // 2
        )

    def forward(self, x):

        # 1. (B, T, C) -> (B, C, T)
        x_transposed = x.transpose(1, 2)

        # 2. Moving Average -> Trend
        trend = self.avg(x_transposed)

        # 3. (B, C, T) -> (B, T, C)
        trend = trend.transpose(1, 2)

        # 4. Với kernel chẵn, AvgPool có thể sinh thêm 1 timestep
        if trend.size(1) != x.size(1):
            trend = trend[:, :x.size(1)]

        # 5. Seasonal = Original - Trend
        seasonal = x - trend

        return trend, seasonal


## 8. Bộ trích xuất đặc trưng dựa trên Autoformer

### 8.1. Mục đích

Class `SimpleAutoformer` xây dựng một backbone dự báo chuỗi thời gian bằng cách kết hợp Series Decomposition với Transformer Encoder.

Kiến trúc gồm hai nhánh xử lý:

- Seasonal branch: xử lý phần dư bằng Transformer.
- Trend branch: xử lý thành phần xu hướng bằng tầng tuyến tính.

### 8.2. Seasonal Branch

Phần dư sau khi phân rã được đưa qua:

1. Linear Embedding.
2. Sin-Cos Positional Encoding.
3. Transformer Encoder.
4. Mean Pooling theo chiều thời gian.
5. Linear Forecasting Head.

Nhánh này trả về vector có kích thước `(B, H)`.

### 8.3. Trend Branch

Thành phần xu hướng được trải phẳng từ `(B, T, C)` thành `(B, T × C)`.

Một tầng tuyến tính ánh xạ vector này sang `(B, H)`.

### 8.4. Kết hợp hai nhánh

Đầu ra cuối cùng được xác định bằng tổng:

$$
\hat{y}=
\hat{y}_{seasonal}+\hat{y}_{trend}
$$

Kích thước đầu ra là `(B, H)`.

### 8.5. Cấu hình

| Thành phần | Giá trị |
|---|---|
| Model dimension | 96 |
| Attention heads | 4 |
| Transformer layers | 3 |
| Feedforward dimension | 64 |
| Decomposition kernel | Patch length |

### 8.6. Giới hạn triển khai

`SimpleAutoformer` sử dụng Transformer Encoder tiêu chuẩn của PyTorch.

Code không triển khai riêng cơ chế Auto-Correlation của Autoformer gốc. Vì vậy, đây là backbone đơn giản hóa dựa trên ý tưởng phân rã chuỗi thời gian.


In [ ]:
class SimpleAutoformer(nn.Module):

    def __init__(
        self,
        input_len,
        horizon,
        input_dim,
        patch_len,
        d_model=96,
        num_heads=4,
        dim_ff=64,
        num_layers=3
    ):
        super().__init__()

        # 1. Series Decomposition
        self.dec = SeriesDecomp(
            k=patch_len
        )

        # 2. Seasonal Branch
        self.embed = nn.Linear(
            input_dim,
            d_model
        )

        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=input_len
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        # 3. Forecasting Heads
        self.fc_seas = nn.Linear(
            d_model,
            horizon
        )

        self.fc_trend = nn.Linear(
            input_len * input_dim,
            horizon
        )

    def forward(self, x):

        # 1. Tách Trend và Seasonal
        trend, seas = self.dec(x)

        # 2. Seasonal Branch
        seas = self.embed(seas)
        seas = self.pos(seas)
        seas = self.encoder(seas)

        # Tổng hợp thông tin theo chiều thời gian
        seas_o = self.fc_seas(
            seas.mean(dim=1)
        )

        # 3. Trend Branch
        trend_o = self.fc_trend(
            trend.reshape(trend.size(0), -1)
        )

        # 4. Kết hợp hai thành phần
        return seas_o + trend_o


## 9. Mô hình kết hợp Autoformer và toán tử Koopman

### 9.1. Kiến trúc

Class `Koopformer_Autoformer` kết hợp:

1. `SimpleAutoformer`: tạo biểu diễn từ chuỗi đầu vào.
2. `StrictStableKoopmanOperator`: thực hiện phép biến đổi ổn định.
3. `Linear Forecasting Head`: tạo dự báo cho sáu kênh.

### 9.2. Biểu diễn tiềm ẩn

Khác với biến thể PatchTST, backbone Autoformer hiện tại trả về vector có số chiều bằng Forecast Horizon:

$$
\mathbf{z}\in\mathbb{R}^{H}
$$

Vì vậy, toán tử Koopman có kích thước:

$$
K\in\mathbb{R}^{H\times H}
$$

Đây là đặc điểm quan trọng của cách triển khai hiện tại.

### 9.3. Giải mã dự báo

Sau phép biến đổi Koopman, tầng tuyến tính ánh xạ:

$$
\mathbb{R}^{H}
\rightarrow
\mathbb{R}^{6H}
$$

Đầu ra chứa các giá trị dự báo của sáu kênh Electricity trong $H$ bước tương lai.

### 9.4. Giá trị trả về

Khi `return_latents=True`, mô hình trả về:

- Dự báo.
- Biểu diễn trước Koopman.
- Biểu diễn sau Koopman.

Các biểu diễn này được sử dụng trong hàm loss có chính quy hóa Lyapunov.

### 9.5. Lưu ý khi so sánh

Kích thước không gian tiềm ẩn của biến thể Autoformer phụ thuộc vào $H$, trong khi PatchTST sử dụng 96 chiều.

Do đó, cần lưu ý sự khác biệt về kiến trúc khi diễn giải kết quả so sánh giữa các mô hình.


In [ ]:
class Koopformer_Autoformer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=96
    ):
        super().__init__()

        # 1. Autoformer Backbone
        self.backbone = SimpleAutoformer(
            input_len=seq_len,
            horizon=horizon,
            input_dim=input_dim,
            patch_len=patch_len,
            d_model=d_model
        )

        # 2. Strict Stable Koopman Operator
        # Latent dimension bằng Forecast Horizon
        self.koop = StrictStableKoopmanOperator(
            latent_dim=horizon
        )

        # 3. Forecasting Head
        self.fc = nn.Linear(
            horizon,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # 1. Trích xuất latent representation
        z = self.backbone(x)

        # 2. Biến đổi bằng Koopman Operator
        z_next, K, Sigma = self.koop(z)

        # 3. Dự báo các nguồn phát điện
        pred = self.fc(z_next)

        if return_latents:
            return pred, z, z_next

        return pred


## 10. Bộ trích xuất đặc trưng dựa trên Informer

### 10.1. Mục đích

Class `InformerSparse` định nghĩa backbone Transformer để tạo biểu diễn tiềm ẩn từ chuỗi thời gian Electricity.

Backbone hỗ trợ hai cách nhúng đầu vào tùy theo Patch Length.

### 10.2. Input Embedding

Khi `patch_len > 1`, mô hình sử dụng `PatchEmbed1D`.

Khi `patch_len = 1`, mô hình sử dụng tầng Linear Embedding để ánh xạ từng timestep sang không gian `d_model`.

Trong cấu hình huấn luyện Electricity hiện tại, Patch Length nằm trong khoảng 70–130 nên nhánh Patch Embedding được sử dụng.

### 10.3. Transformer Encoder

Sau khi nhúng đầu vào, mô hình thực hiện:

1. Bổ sung Sin-Cos Positional Encoding.
2. Biến đổi bằng Transformer Encoder.
3. Mean Pooling theo chiều token.

### 10.4. Cấu hình

| Tham số | Giá trị |
|---|---|
| Model dimension | 96 |
| Attention heads | 4 |
| Transformer layers | 3 |
| Feedforward dimension | 96 |
| Output dimension | 96 |

### 10.5. Giới hạn triển khai

Mặc dù class có tên `InformerSparse`, code sử dụng `nn.TransformerEncoder` tiêu chuẩn.

Không có cơ chế ProbSparse Attention được cài đặt riêng.

Vì vậy, backbone này chỉ nên được mô tả là biến thể Transformer dựa trên Informer, không phải bản tái hiện đầy đủ Informer gốc.


In [ ]:
class InformerSparse(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=96,
        num_heads=4,
        dim_ff=96,
        num_layers=3,
        seq_len=120,
        patch_len=1
    ):
        super().__init__()

        # 1. Có sử dụng Patch Embedding hay không?
        self.use_patch = patch_len > 1

        if self.use_patch:

            self.patch = PatchEmbed1D(
                in_ch=input_dim,
                d_model=d_model,
                patch_len=patch_len,
                stride=patch_len
            )

            n_tokens = int(np.ceil(seq_len / patch_len))

        else:

            self.embed = nn.Linear(
                input_dim,
                d_model
            )

            n_tokens = seq_len

        # 2. Positional Encoding
        self.pos = SinCosPosEnc(
            d_model=d_model,
            max_len=n_tokens
        )

        # 3. Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=dim_ff,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

    def forward(self, x):

        # 1. Input Embedding
        if self.use_patch:
            x = self.patch(x)
        else:
            x = self.embed(x)

        # 2. Positional Encoding
        x = self.pos(x)

        # 3. Transformer Encoder
        x = self.encoder(x)

        # 4. Mean Pooling
        return x.mean(dim=1)


## 11. Mô hình kết hợp Informer và toán tử Koopman

### 11.1. Kiến trúc tổng thể

Class `Koopformer_Informer` kết hợp:

- Backbone `InformerSparse`.
- Toán tử Koopman ổn định nghiêm ngặt.
- Bộ giải mã dự báo tuyến tính.

Luồng xử lý:

**Input → Informer-based Backbone → Koopman Operator → Forecasting Head → Prediction**

### 11.2. Biểu diễn tiềm ẩn

Backbone trả về biểu diễn có 96 chiều:

$$
\mathbf{z}\in\mathbb{R}^{96}
$$

Toán tử Koopman thực hiện phép biến đổi:

$$
\mathbf{z}_{next}=K\mathbf{z}
$$

Với $K\in\mathbb{R}^{96\times96}$.

### 11.3. Đầu ra dự báo

Bộ giải mã tuyến tính ánh xạ vector tiềm ẩn thành vector gồm $6H$ giá trị, tương ứng với sáu kênh và $H$ bước dự báo.

### 11.4. Giá trị trả về

Chế độ thông thường trả về dự báo.

Chế độ `return_latents=True` trả về dự báo và hai biểu diễn tiềm ẩn trước, sau phép biến đổi Koopman.

Các biểu diễn này phục vụ tính toán thành phần chính quy hóa Lyapunov trong huấn luyện.


In [ ]:
class Koopformer_Informer(nn.Module):

    def __init__(
        self,
        input_dim,
        seq_len,
        horizon,
        patch_len,
        d_model=96
    ):
        super().__init__()

        # 1. Informer Backbone
        self.backbone = InformerSparse(
            input_dim=input_dim,
            d_model=d_model,
            seq_len=seq_len,
            patch_len=patch_len
        )

        # 2. Strict Stable Koopman Operator
        self.koop = StrictStableKoopmanOperator(
            latent_dim=d_model
        )

        # 3. Forecasting Head
        self.fc = nn.Linear(
            d_model,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # 1. Extract latent representation
        z = self.backbone(x)

        # 2. Apply Koopman transformation
        z_next, K, Sigma = self.koop(z)

        # 3. Forecast future values
        pred = self.fc(z_next)

        # 4. Return latent representations if needed
        if return_latents:
            return pred, z, z_next

        return pred


## 12. Mô hình đối chứng LSTM

### 12.1. Mục đích

Class `SimpleLSTMForecaster` được sử dụng làm mô hình đối chứng trong thực nghiệm Electricity.

Khác với các biến thể Koopformer, LSTM trực tiếp xử lý chuỗi thời gian bằng mạng hồi quy tuần tự và dự báo bằng một tầng tuyến tính.

### 12.2. Kiến trúc

Mô hình gồm:

1. LSTM nhiều tầng.
2. Hidden State của tầng cuối.
3. Linear Forecasting Head.

LSTM nhận dữ liệu có kích thước:

`(batch_size, sequence_length, 6)`

Sau bước xử lý tuần tự, hidden state của tầng cuối được sử dụng làm biểu diễn tổng hợp.

### 12.3. Cấu hình

| Tham số | Giá trị |
|---|---|
| Input dimension | 6 |
| Hidden dimension | 48 |
| LSTM layers | 2 |
| Forecasting head | Linear |
| Output dimension | 6 × Horizon |

### 12.4. Dự báo

Hidden state cuối được đưa vào tầng tuyến tính để tạo vector dự báo có $6H$ phần tử.

Mô hình không sử dụng toán tử Koopman.

### 12.5. Giá trị trả về

Khi `return_latents=True`, code trả về `pred, z, z`.

Hai biểu diễn `z` giống nhau, vì vậy thành phần chính quy hóa dựa trên sự gia tăng năng lượng giữa chúng bằng không.

LSTM đóng vai trò baseline để đối chiếu kết quả dự báo với ba biến thể Koopformer.


In [ ]:
class SimpleLSTMForecaster(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        num_layers,
        horizon
    ):
        super().__init__()

        # 1. LSTM Layers
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True
        )

        # 2. Forecasting Head
        self.fc = nn.Linear(
            hidden_dim,
            horizon * input_dim
        )

    def forward(self, x, return_latents=False):

        # 1. LSTM Forward Pass
        _, (h_n, c_n) = self.lstm(x)

        # 2. Lấy hidden state của layer cuối
        z = h_n[-1]

        # 3. Forecasting
        pred = self.fc(z)

        # 4. Trả về latent nếu cần
        if return_latents:
            return pred, z, z

        return pred


## 13. Hàm tạo cửa sổ trượt

### 13.1. Mục đích

Hàm `create_windows()` chuyển dữ liệu chuỗi thời gian đã chuẩn hóa thành các cặp input và target để huấn luyện, đánh giá mô hình.

### 13.2. Phương pháp

Với:

- $N$: số timestamps.
- $P$: độ dài cửa sổ đầu vào.
- $H$: Forecast Horizon.
- $C$: số kênh dữ liệu.

Mỗi mẫu đầu vào có kích thước:

$$
X_i\in\mathbb{R}^{P\times C}
$$

Target gồm $H$ bước tương lai của $C$ kênh, được chuyển vị và trải phẳng:

$$
y_i\in\mathbb{R}^{H C}
$$

### 13.3. Số lượng cửa sổ

Mã nguồn hiện tại sử dụng:

`range(len(data) - seq_len - horizon)`

Do đó số lượng mẫu được tạo là:

$$
N_{\mathrm{windows}}=N-P-H
$$

Quy tắc đánh chỉ số này được giữ nguyên để tương thích với dữ liệu và kết quả thực nghiệm đã có.

### 13.4. Kết quả

Hàm trả về hai PyTorch Tensor:

- `X`: `(N_windows, P, C)`.
- `y`: `(N_windows, H × C)`.

Hàm được sử dụng lại trong notebook Training.


In [ ]:
def create_windows(data, seq_len, horizon):

    X, y = [], []

    for i in range(len(data) - seq_len - horizon):

        # Input: seq_len historical timestamps
        x_window = data[
            i : i + seq_len
        ]

        # Target: horizon future timestamps
        y_window = data[
            i + seq_len : i + seq_len + horizon
        ].T.reshape(-1)

        X.append(x_window)
        y.append(y_window)

    # Convert to PyTorch tensors
    X = torch.stack(X).float()
    y = torch.stack(y).float()

    return X, y


## 14. Hàm mất mát và chính quy hóa Lyapunov

### 14.1. Mục đích

Hàm `compute_loss()` kết hợp sai số dự báo với thành phần chính quy hóa Lyapunov nhằm hạn chế sự gia tăng năng lượng trong không gian tiềm ẩn.

### 14.2. Sai số dự báo

Thành phần dự báo sử dụng Mean Squared Error:

$$
\mathcal{L}_{MSE}
=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)^2
$$

### 14.3. Hàm năng lượng Lyapunov

Mã nguồn sử dụng hàm năng lượng bậc hai:

$$
V(z)=z^\top Pz
$$

Với $P=I$, ta có:

$$
V(z)=\|z\|_2^2
$$

Mức thay đổi năng lượng sau phép biến đổi Koopman được xác định bởi:

$$
\Delta V=V(z_{next})-V(z)
$$

Thành phần phạt chỉ áp dụng khi năng lượng tăng:

$$
\mathcal{L}_{Lyap}
=
\operatorname{mean}
\left[
\max(0,\Delta V)
\right]
$$

### 14.4. Hàm mất mát tổng

$$
\mathcal{L}_{total}
=
\mathcal{L}_{MSE}
+
\lambda\mathcal{L}_{Lyap}
$$

Trong đó hệ số chính quy hóa mặc định là:

$$
\lambda=0.1
$$

### 14.5. Giá trị trả về

Hàm trả về:

- `total_loss`.
- `mse_loss`.
- `lyap_loss`.

Những giá trị này được sử dụng trong quá trình huấn luyện và theo dõi sự hội tụ.


In [ ]:
def compute_loss(
    pred,
    target,
    z,
    z_next,
    lyap_weight=0.1
):

    # 1. Prediction Loss
    mse_loss = nn.functional.mse_loss(
        pred,
        target
    )

    # 2. Lyapunov Regularization
    if lyap_weight > 0:

        # Identity matrix
        P = torch.eye(
            z.shape[1],
            device=z.device,
            dtype=z.dtype
        )

        # V(z) = z^T P z
        V_z = torch.einsum(
            "bi,ij,bj->b",
            z, P, z
        )

        # V(z_next) = z_next^T P z_next
        V_z_next = torch.einsum(
            "bi,ij,bj->b",
            z_next, P, z_next
        )

        # Penalize positive increase
        lyap_loss = torch.relu(
            V_z_next - V_z
        ).mean()

    else:

        lyap_loss = torch.zeros_like(
            mse_loss
        )

    # 3. Total Loss
    total_loss = (
        mse_loss
        + lyap_weight * lyap_loss
    )

    return total_loss, mse_loss, lyap_loss


## 15. Hàm huấn luyện mô hình

### 15.1. Mục đích

Hàm `train_model()` định nghĩa quy trình tối ưu tham số mô hình từ dữ liệu huấn luyện.

Đây là định nghĩa hàm được notebook Training tái sử dụng, không phải một lần huấn luyện được thực thi trực tiếp trong notebook Model.

### 15.2. Cấu hình

| Tham số | Giá trị mặc định |
|---|---|
| Epochs | 4000 |
| Optimizer | Adam |
| Learning rate | 0.0003 |
| Lyapunov weight | 0.1 |
| Training batch | Toàn bộ dữ liệu được truyền vào hàm |

### 15.3. Quy trình huấn luyện

Trong mỗi epoch:

1. Chuyển mô hình sang chế độ Training.
2. Xóa gradient đã tích lũy.
3. Thực hiện Forward Pass.
4. Tính MSE và Lyapunov Loss.
5. Thực hiện Backpropagation.
6. Cập nhật tham số bằng Adam.
7. Lưu lịch sử các giá trị loss.

### 15.4. Các đại lượng được theo dõi

Hàm lưu:

- `loss`: Total Loss.
- `mse`: Prediction MSE.
- `lyap`: Lyapunov Loss.
- `eig`: đại lượng theo dõi từ tham số của toán tử Koopman.

Trong code hiện tại, `eig` được tính từ giá trị lớn nhất của `sigmoid(S_raw)` mà chưa nhân với `rho_max`.

Vì vậy, không nên diễn giải trực tiếp `eig` là giá trị riêng lớn nhất hay chuẩn phổ thực tế của ma trận Koopman.

### 15.5. Kết quả trả về

Hàm trả về mô hình sau huấn luyện cùng các lịch sử loss để phục vụ đánh giá và tổng hợp kết quả.

Notebook này chỉ định nghĩa hàm, không gọi huấn luyện.


In [ ]:
def train_model(
    model,
    X_train,
    y_train,
    epochs=4000,
    lr=3e-4,
    lyap_weight=0.1,
    verbose=True
):

    # 1. Move model and data to device
    model = model.to(DEVICE)

    X_train = X_train.to(DEVICE)
    y_train = y_train.to(DEVICE)

    # 2. Optimizer
    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    # 3. Training histories
    loss_history = []
    mse_history = []
    lyap_history = []
    eig_history = []

    log_interval = max(1, epochs // 10)

    # 4. Training loop
    for epoch in range(epochs):

        model.train()

        # Reset gradients
        optimizer.zero_grad()

        # Forward pass
        pred, z, z_next = model(
            X_train,
            return_latents=True
        )

        # Compute loss
        total_loss, mse_loss, lyap_loss = compute_loss(
            pred=pred,
            target=y_train,
            z=z,
            z_next=z_next,
            lyap_weight=lyap_weight
        )

        # Backpropagation
        total_loss.backward()

        # Update parameters
        optimizer.step()

        # 5. Save training histories
        loss_history.append(
            total_loss.item()
        )

        mse_history.append(
            mse_loss.item()
        )

        lyap_history.append(
            lyap_loss.item()
        )

        # 6. Track Koopman quantity
        if hasattr(model, "koop"):

            with torch.no_grad():

                eig_max = (
                    torch.sigmoid(model.koop.S_raw)
                    .abs()
                    .max()
                    .item()
                )

        else:
            eig_max = 0.0

        eig_history.append(eig_max)

        # 7. Print training progress
        if verbose and (
            epoch % log_interval == 0
            or epoch == epochs - 1
        ):

            print(
                f"Epoch {epoch+1:4d}/{epochs} | "
                f"MSE: {mse_loss.item():.6f} | "
                f"Lyap: {lyap_loss.item():.6f} | "
                f"Total: {total_loss.item():.6f}"
            )

    # 8. Return training results
    return {
        "model": model,
        "loss": np.array(loss_history),
        "mse": np.array(mse_history),
        "lyap": np.array(lyap_history),
        "eig": np.array(eig_history)
    }


## 16. Hàm đánh giá mô hình

### 16.1. Mục đích

Hàm `evaluate_model()` đánh giá chất lượng dự báo của mô hình trên tập dữ liệu được cung cấp.

Quá trình đánh giá không cập nhật trọng số mô hình.

### 16.2. Quy trình

1. Chuyển mô hình sang chế độ `eval()`.
2. Tắt gradient bằng `torch.no_grad()`.
3. Chia dữ liệu thành các batch để suy luận.
4. Tổng hợp các giá trị dự báo.
5. So sánh dự báo với target thực tế.

Batch size mặc định là 256.

### 16.3. Chỉ số đánh giá

Hai metric được sử dụng:

**Mean Squared Error (MSE):**

$$
MSE=\frac{1}{N}\sum_{i=1}^{N}(\hat{y}_i-y_i)^2
$$

**Mean Absolute Error (MAE):**

$$
MAE=\frac{1}{N}\sum_{i=1}^{N}|\hat{y}_i-y_i|
$$

### 16.4. Kết quả trả về

Hàm trả về dictionary gồm:

- `MSE`: sai số bình phương trung bình.
- `MAE`: sai số tuyệt đối trung bình.
- `predictions`: tensor chứa các dự báo.

Các kết quả được tính trên tập dữ liệu truyền vào hàm, có thể là Train hoặc Test tùy theo cách gọi trong notebook Training.


In [ ]:
def evaluate_model(
    model,
    X,
    y,
    batch_size=256
):

    # 1. Evaluation mode
    model.eval()

    device = next(
        model.parameters()
    ).device

    all_predictions = []

    # 2. Disable gradients
    with torch.no_grad():

        for i in range(
            0,
            len(X),
            batch_size
        ):

            x_batch = X[
                i : i + batch_size
            ].to(device)

            # Model predictions
            pred = model(x_batch)

            all_predictions.append(
                pred.cpu()
            )

    # 3. Concatenate predictions
    predictions = torch.cat(
        all_predictions,
        dim=0
    )

    # 4. Targets on CPU
    targets = y.cpu()

    # 5. Evaluation metrics
    mse = torch.mean(
        (predictions - targets) ** 2
    ).item()

    mae = torch.mean(
        torch.abs(predictions - targets)
    ).item()

    return {
        "MSE": mse,
        "MAE": mae,
        "predictions": predictions
    }


## 17. Khởi tạo mô hình theo cấu hình thực nghiệm

### 17.1. Mục đích

Hàm `create_model()` cung cấp giao diện thống nhất để khởi tạo các mô hình sử dụng trong Grid Search.

Hàm nhận ba tham số:

- `model_name`: tên mô hình.
- `patch_len`: độ dài chuỗi đầu vào.
- `horizon`: số bước dự báo.

### 17.2. Các mô hình được hỗ trợ

| Tên mô hình | Class |
|---|---|
| LSTM | `SimpleLSTMForecaster` |
| PatchTST | `Koopformer_PatchTST` |
| Autoformer | `Koopformer_Autoformer` |
| Informer | `Koopformer_Informer` |

### 17.3. Cấu hình được sử dụng

Tất cả mô hình nhận cùng số kênh đầu vào là 6.

Các biến thể Koopformer sử dụng `d_model=96`.

LSTM sử dụng hidden dimension bằng 48 và hai tầng LSTM.

Forecast Horizon quyết định số lượng giá trị được dự báo.

### 17.4. Vai trò trong Grid Search

Hàm `create_model()` được notebook Training gọi để tạo từng mô hình tương ứng với các cấu hình Patch Length và Forecast Horizon.

Mô hình được chuyển đến thiết bị tính toán đã chọn thông qua `.to(DEVICE)`.

### 17.5. Phạm vi

Hàm chỉ khởi tạo mô hình và không thực hiện huấn luyện, đánh giá hoặc lưu checkpoint.


In [ ]:
def create_model(
    model_name,
    patch_len,
    horizon
):

    if model_name == "LSTM":

        model = SimpleLSTMForecaster(
            input_dim=INPUT_DIM,
            hidden_dim=LSTM_HIDDEN_DIM,
            num_layers=2,
            horizon=horizon
        )

    elif model_name == "PatchTST":

        model = Koopformer_PatchTST(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    elif model_name == "Autoformer":

        model = Koopformer_Autoformer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    elif model_name == "Informer":

        model = Koopformer_Informer(
            input_dim=INPUT_DIM,
            seq_len=patch_len,
            horizon=horizon,
            patch_len=patch_len,
            d_model=D_MODEL
        )

    else:
        raise ValueError(
            f"Unknown model: {model_name}"
        )

    return model.to(DEVICE)
    